# CNN Training — သိကိုသိရမယ့် Concepts

CNN model ကို train လုပ်တဲ့ process တစ်ခုလုံးကို **တစ်ဆင့်ချင်းစီ** number examples နဲ့ တွက်ပြမယ်။

## Training Flow Overview

```
┌─────────────────────────────────────────────────────┐
│  1. Forward Pass  →  Input ကနေ Prediction ထုတ်       │
│  2. Loss          →  Prediction vs Target ဘယ်လောက်ကွာလဲ │
│  3. Backward Pass →  Gradient (slope) တွက်           │
│  4. Update Weights→  Gradient Descent နဲ့ weights ပြင် │
│  5. Repeat        →  Loss နည်းသွားအောင် ထပ်ခါထပ်ခါ    │
└─────────────────────────────────────────────────────┘
```

## Contents
1. **Forward Pass** — input → model → prediction
2. **Loss Function** — prediction ဘယ်လောက် မှားလဲ တိုင်းတာ
3. **Backpropagation** — gradient (slope) တွက်ချက်ခြင်း
4. **Gradient Descent & Optimizers** — weights update လုပ်ခြင်း
5. **Learning Rate & Scheduler** — update step size ထိန်းခြင်း
6. **Overfitting & Regularization** — model memorize မလုပ်အောင် ထိန်းခြင်း
7. **Full Training Loop** — အဆင့်အကုန် ပေါင်းပြီး train ခြင်း

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

Matplotlib is building the font cache; this may take a moment.


---

## 1. Forward Pass — Input ကနေ Prediction ထုတ်ခြင်း

Model ထဲကို input image ထည့်ပြီး **layer by layer** ဖြတ်သွားတယ်။

Simple example: **1 Conv layer + 1 FC layer** mini CNN

```
Input (1x4x4)                    Conv2d(1→1, 3x3)
┌──────────────┐                 ┌─────────┐
│ 1  2  3  4   │                 │ 0  1  0 │
│ 5  6  7  8   │  ──conv2d──►   │ 1  1  1 │  (kernel/filter)
│ 9  10 11 12  │                 │ 0  1  0 │
│ 13 14 15 16  │                 └─────────┘
└──────────────┘
```

Convolution = kernel ကို input ပေါ် slide ပြီး **element-wise multiply & sum**

In [2]:
# --- Forward Pass: Hand-calculated Example ---

# Input: 1 channel, 4x4 image
input_img = torch.tensor([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9,  10, 11, 12],
    [13, 14, 15, 16]
], dtype=torch.float32).reshape(1, 1, 4, 4)  # (N=1, C=1, H=4, W=4)

# Small CNN: Conv2d → ReLU → Flatten → Linear
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)

# kernel weights ကို manually set (ရှင်းပြဖို့)
with torch.no_grad():
    conv.weight.copy_(torch.tensor([
        [0, 1, 0],
        [1, 1, 1],
        [0, 1, 0]
    ], dtype=torch.float32).reshape(1, 1, 3, 3))

print("=== Step 1: Convolution ===")
print(f"Input (4x4):\n{input_img.squeeze()}\n")
print(f"Kernel (3x3):\n{conv.weight.squeeze()}\n")

# === Hand calculation ===
# Conv output: (4-3+1) = 2x2 output (no padding)
# Position (0,0): 0*1 + 1*2 + 0*3 + 1*5 + 1*6 + 1*7 + 0*9 + 1*10 + 0*11
#               = 0 + 2 + 0 + 5 + 6 + 7 + 0 + 10 + 0 = 30
# Position (0,1): 0*2 + 1*3 + 0*4 + 1*6 + 1*7 + 1*8 + 0*10 + 1*11 + 0*12
#               = 0 + 3 + 0 + 6 + 7 + 8 + 0 + 11 + 0 = 35
# Position (1,0): 0*5 + 1*6 + 0*7 + 1*9 + 1*10 + 1*11 + 0*13 + 1*14 + 0*15
#               = 0 + 6 + 0 + 9 + 10 + 11 + 0 + 14 + 0 = 50
# Position (1,1): 0*6 + 1*7 + 0*8 + 1*10 + 1*11 + 1*12 + 0*14 + 1*15 + 0*16
#               = 0 + 7 + 0 + 10 + 11 + 12 + 0 + 15 + 0 = 55

conv_out = conv(input_img)
print(f"Conv output (2x2):\n{conv_out.squeeze()}")
print()
print("Hand calculation:")
print("  (0,0): 0×1 + 1×2 + 0×3 + 1×5 + 1×6 + 1×7 + 0×9 + 1×10 + 0×11 = 30")
print("  (0,1): 0×2 + 1×3 + 0×4 + 1×6 + 1×7 + 1×8 + 0×10 + 1×11 + 0×12 = 35")
print("  (1,0): 0×5 + 1×6 + 0×7 + 1×9 + 1×10 + 1×11 + 0×13 + 1×14 + 0×15 = 50")
print("  (1,1): 0×6 + 1×7 + 0×8 + 1×10 + 1×11 + 1×12 + 0×14 + 1×15 + 0×16 = 55")

=== Step 1: Convolution ===
Input (4x4):
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])

Kernel (3x3):
tensor([[0., 1., 0.],
        [1., 1., 1.],
        [0., 1., 0.]], grad_fn=<SqueezeBackward0>)

Conv output (2x2):
tensor([[30., 35.],
        [50., 55.]], grad_fn=<SqueezeBackward0>)

Hand calculation:
  (0,0): 0×1 + 1×2 + 0×3 + 1×5 + 1×6 + 1×7 + 0×9 + 1×10 + 0×11 = 30
  (0,1): 0×2 + 1×3 + 0×4 + 1×6 + 1×7 + 1×8 + 0×10 + 1×11 + 0×12 = 35
  (1,0): 0×5 + 1×6 + 0×7 + 1×9 + 1×10 + 1×11 + 0×13 + 1×14 + 0×15 = 50
  (1,1): 0×6 + 1×7 + 0×8 + 1×10 + 1×11 + 1×12 + 0×14 + 1×15 + 0×16 = 55


In [3]:
# --- Forward Pass: ReLU → Flatten → Linear ---
print("=== Step 2: ReLU Activation ===")
relu_out = torch.relu(conv_out)
print(f"ReLU output:\n{relu_out.squeeze()}")
print("(negative values → 0, positive values → unchanged)")
print("(ဒီ example မှာ values အကုန် positive ဖြစ်လို့ ပြောင်းလဲမှု မရှိ)")
print()

print("=== Step 3: Flatten ===")
flat = relu_out.view(1, -1)  # (1, 4)
print(f"Flatten: {relu_out.squeeze().shape} → {flat.shape}")
print(f"Values: {flat.squeeze().tolist()}")
print()

print("=== Step 4: Linear (FC) Layer ===")
# 4 inputs → 3 outputs (eg. 3 classes: cat, dog, bird)
fc = nn.Linear(4, 3, bias=False)
with torch.no_grad():
    fc.weight.copy_(torch.tensor([
        [0.1, 0.2, -0.1, 0.3],   # class 0 weights
        [-0.2, 0.1, 0.4, -0.1],  # class 1 weights
        [0.3, -0.1, 0.2, 0.1],   # class 2 weights
    ]))

logits = fc(flat)
print(f"Weights (3x4):\n{fc.weight}")
print()

# Hand calculation:
# class 0: 0.1×30 + 0.2×35 + (-0.1)×50 + 0.3×55 = 3 + 7 - 5 + 16.5 = 21.5
# class 1: (-0.2)×30 + 0.1×35 + 0.4×50 + (-0.1)×55 = -6 + 3.5 + 20 - 5.5 = 12.0
# class 2: 0.3×30 + (-0.1)×35 + 0.2×50 + 0.1×55 = 9 - 3.5 + 10 + 5.5 = 21.0
print(f"Logits (raw scores): {logits.squeeze().tolist()}")
print()
print("Hand calculation:")
print("  class 0: 0.1×30 + 0.2×35 + (-0.1)×50 + 0.3×55 = 21.5")
print("  class 1: (-0.2)×30 + 0.1×35 + 0.4×50 + (-0.1)×55 = 12.0")
print("  class 2: 0.3×30 + (-0.1)×35 + 0.2×50 + 0.1×55 = 21.0")
print()

# Softmax → probability
probs = torch.softmax(logits, dim=1)
print(f"Softmax probabilities: {[f'{p:.4f}' for p in probs.squeeze().tolist()]}")
print(f"Predicted class: {probs.argmax().item()} (highest probability)")

=== Step 2: ReLU Activation ===
ReLU output:
tensor([[30., 35.],
        [50., 55.]], grad_fn=<SqueezeBackward0>)
(negative values → 0, positive values → unchanged)
(ဒီ example မှာ values အကုန် positive ဖြစ်လို့ ပြောင်းလဲမှု မရှိ)

=== Step 3: Flatten ===
Flatten: torch.Size([2, 2]) → torch.Size([1, 4])
Values: [30.0, 35.0, 50.0, 55.0]

=== Step 4: Linear (FC) Layer ===
Weights (3x4):
Parameter containing:
tensor([[ 0.1000,  0.2000, -0.1000,  0.3000],
        [-0.2000,  0.1000,  0.4000, -0.1000],
        [ 0.3000, -0.1000,  0.2000,  0.1000]], requires_grad=True)

Logits (raw scores): [21.5, 12.0, 21.0]

Hand calculation:
  class 0: 0.1×30 + 0.2×35 + (-0.1)×50 + 0.3×55 = 21.5
  class 1: (-0.2)×30 + 0.1×35 + 0.4×50 + (-0.1)×55 = 12.0
  class 2: 0.3×30 + (-0.1)×35 + 0.2×50 + 0.1×55 = 21.0

Softmax probabilities: ['0.6224', '0.0000', '0.3775']
Predicted class: 0 (highest probability)


---

## 2. Loss Function — Prediction ဘယ်လောက်မှားလဲ တိုင်းတာ

Model ရဲ့ prediction က target (ground truth) နဲ့ **ဘယ်လောက်ကွာ**လဲ ဆိုတာကို number တစ်ခုအနေနဲ့ ပြောပေးတယ်။

### Cross-Entropy Loss (Classification Standard)

$$\text{CrossEntropyLoss} = -\log\left(\frac{e^{z_y}}{\sum_{j} e^{z_j}}\right)$$

- $z$ = logits (raw scores from FC layer)
- $y$ = true class index
- **Softmax + Negative Log Likelihood** ကို combine ထားတာ

### Example တွက်ကြည့်မယ်:
- Logits = [21.5, 12.0, 21.0]
- True label = class 0
- Softmax → probabilities → `-log(P(class_0))`

In [4]:
# --- Loss Function: Hand-calculated Cross-Entropy ---

# logits from forward pass
logits_example = torch.tensor([[21.5, 12.0, 21.0]])  # raw scores
true_label = torch.tensor([0])  # true class = 0

print("=== Cross-Entropy Loss: Step by Step ===\n")
print(f"Logits:     {logits_example.squeeze().tolist()}")
print(f"True label: {true_label.item()} (class 0)")
print()

# Step 1: Softmax
# P(class_i) = e^z_i / Σ(e^z_j)
z = logits_example.squeeze()
exp_z = torch.exp(z)
softmax_probs = exp_z / exp_z.sum()

print("Step 1: Softmax")
print(f"  e^21.5 = {exp_z[0]:.2f}")
print(f"  e^12.0 = {exp_z[1]:.2f}")
print(f"  e^21.0 = {exp_z[2]:.2f}")
print(f"  Sum    = {exp_z.sum():.2f}")
print(f"  P(class 0) = {exp_z[0]:.2f} / {exp_z.sum():.2f} = {softmax_probs[0]:.6f}")
print(f"  P(class 1) = {exp_z[1]:.2f} / {exp_z.sum():.2f} = {softmax_probs[1]:.6f}")
print(f"  P(class 2) = {exp_z[2]:.2f} / {exp_z.sum():.2f} = {softmax_probs[2]:.6f}")
print()

# Step 2: Negative Log Likelihood
# Loss = -log(P(true_class))
loss_manual = -torch.log(softmax_probs[0])
print(f"Step 2: -log(P(class 0)) = -log({softmax_probs[0]:.6f}) = {loss_manual:.6f}")
print()

# PyTorch CrossEntropyLoss (same result)
criterion = nn.CrossEntropyLoss()
loss_pytorch = criterion(logits_example, true_label)
print(f"PyTorch CrossEntropyLoss: {loss_pytorch.item():.6f}")
print(f"Match: {torch.allclose(loss_manual, loss_pytorch)}")
print()

# --- Loss interpretation ---
print("=== Loss Interpretation ===")
print("  Loss = 0      → perfect prediction (P(true)=1)")
print("  Loss ↓ small  → model confident & correct")
print("  Loss ↑ large  → model wrong / not confident")
print()

# --- Different scenarios ---
print("=== Loss Scenarios ===")
scenarios = [
    ("Confident & Correct", torch.tensor([[10.0, 0.0, 0.0]]), torch.tensor([0])),
    ("Uncertain",           torch.tensor([[1.0, 1.0, 1.0]]),   torch.tensor([0])),
    ("Confident & Wrong",   torch.tensor([[0.0, 0.0, 10.0]]),  torch.tensor([0])),
]
for name, logit, label in scenarios:
    loss = criterion(logit, label)
    prob = torch.softmax(logit, dim=1)[0, label.item()]
    print(f"  {name:25s} → P(true)={prob:.4f}, Loss={loss.item():.4f}")

=== Cross-Entropy Loss: Step by Step ===

Logits:     [21.5, 12.0, 21.0]
True label: 0 (class 0)

Step 1: Softmax
  e^21.5 = 2174359552.00
  e^12.0 = 162754.80
  e^21.0 = 1318815744.00
  Sum    = 3493338112.00
  P(class 0) = 2174359552.00 / 3493338112.00 = 0.622430
  P(class 1) = 162754.80 / 3493338112.00 = 0.000047
  P(class 2) = 1318815744.00 / 3493338112.00 = 0.377523

Step 2: -log(P(class 0)) = -log(0.622430) = 0.474124

PyTorch CrossEntropyLoss: 0.474124
Match: True

=== Loss Interpretation ===
  Loss = 0      → perfect prediction (P(true)=1)
  Loss ↓ small  → model confident & correct
  Loss ↑ large  → model wrong / not confident

=== Loss Scenarios ===
  Confident & Correct       → P(true)=0.9999, Loss=0.0001
  Uncertain                 → P(true)=0.3333, Loss=1.0986
  Confident & Wrong         → P(true)=0.0000, Loss=10.0001


---

## 3. Backpropagation — Gradient (Slope) တွက်ခြင်း

Loss ကို **weight တစ်ခုချင်းစီ** ဘယ်လို affect လဲ ဆိုတဲ့ **gradient** ကို chain rule သုံးပြီး **output ကနေ input ဖက်ကို** ပြန်တွက်ချတယ်။

$$\frac{\partial \text{Loss}}{\partial w} = \text{gradient} = \text{"ဒီ weight ကို နည်းနည်း ပြောင်းရင် Loss ဘယ်လောက် ပြောင်းမလဲ"}$$

### Chain Rule Example

```
Input x → [×w] → z → [ReLU] → a → [×v] → output → Loss
                                              │
        ∂L/∂w = ∂L/∂output × ∂output/∂a × ∂a/∂z × ∂z/∂w
                 ◄────────── chain rule: ပြန်ခြွှတ်တယ် ──────────►
```

### Gradient ရဲ့ Meaning
- **Gradient > 0:** weight ↑ ရင် Loss ↑ → weight ကို **လျှော့ရမယ်**
- **Gradient < 0:** weight ↑ ရင် Loss ↓ → weight ကို **တိုးရမယ်**
- **Gradient ≈ 0:** weight ပြောင်းလည်း Loss မပြောင်း → **plateau/saddle**

In [5]:
# --- Backpropagation: Simple Example ---
# y = w * x + b  →  Loss = (y - target)^2

print("=== Simple Backprop Example ===")
print("Formula: y = w*x + b, Loss = (y - target)²\n")

# Manual values
w = torch.tensor(2.0, requires_grad=True)   # weight
b = torch.tensor(1.0, requires_grad=True)   # bias
x_val = torch.tensor(3.0)                    # input
target = torch.tensor(10.0)                  # target

# Forward pass
y = w * x_val + b           # 2*3 + 1 = 7
loss = (y - target) ** 2    # (7 - 10)² = 9

print(f"Forward Pass:")
print(f"  w={w.item()}, x={x_val.item()}, b={b.item()}")
print(f"  y = w*x + b = {w.item()}×{x_val.item()} + {b.item()} = {y.item()}")
print(f"  Loss = (y - target)² = ({y.item()} - {target.item()})² = {loss.item()}")
print()

# Backward pass — PyTorch autograd
loss.backward()

print(f"Backpropagation (Chain Rule):")
print(f"  ∂Loss/∂y = 2(y - target) = 2({y.item()} - {target.item()}) = {2*(y.item()-target.item())}")
print(f"  ∂y/∂w    = x = {x_val.item()}")
print(f"  ∂y/∂b    = 1")
print()
print(f"  ∂Loss/∂w = ∂Loss/∂y × ∂y/∂w = {2*(y.item()-target.item())} × {x_val.item()} = {w.grad.item()}")
print(f"  ∂Loss/∂b = ∂Loss/∂y × ∂y/∂b = {2*(y.item()-target.item())} × 1 = {b.grad.item()}")
print()
print(f"PyTorch autograd:")
print(f"  w.grad = {w.grad.item()}")
print(f"  b.grad = {b.grad.item()}")
print()
print("✅ gradient negative → weight ကိုတိုးရမယ် (Loss ကျအောင်)")
print("   gradient = -18 → w ကို တိုးရင် Loss ကျမယ် ✓")

=== Simple Backprop Example ===
Formula: y = w*x + b, Loss = (y - target)²

Forward Pass:
  w=2.0, x=3.0, b=1.0
  y = w*x + b = 2.0×3.0 + 1.0 = 7.0
  Loss = (y - target)² = (7.0 - 10.0)² = 9.0

Backpropagation (Chain Rule):
  ∂Loss/∂y = 2(y - target) = 2(7.0 - 10.0) = -6.0
  ∂y/∂w    = x = 3.0
  ∂y/∂b    = 1

  ∂Loss/∂w = ∂Loss/∂y × ∂y/∂w = -6.0 × 3.0 = -18.0
  ∂Loss/∂b = ∂Loss/∂y × ∂y/∂b = -6.0 × 1 = -6.0

PyTorch autograd:
  w.grad = -18.0
  b.grad = -6.0

✅ gradient negative → weight ကိုတိုးရမယ် (Loss ကျအောင်)
   gradient = -18 → w ကို တိုးရင် Loss ကျမယ် ✓


In [6]:
# --- CNN Backprop: Gradients ကို စစ်ကြည့် ---
print("=== CNN Layer Gradients ===\n")

# Mini CNN with tracked gradients
conv_demo = nn.Conv2d(1, 1, kernel_size=3, bias=False)
fc_demo = nn.Linear(4, 3, bias=False)

# Set known weights
with torch.no_grad():
    conv_demo.weight.copy_(torch.tensor([[[[0,1,0],[1,1,1],[0,1,0]]]], dtype=torch.float32))
    fc_demo.weight.copy_(torch.tensor([[0.1,0.2,-0.1,0.3],[-0.2,0.1,0.4,-0.1],[0.3,-0.1,0.2,0.1]]))

# Forward pass
x_demo = input_img.clone().requires_grad_(True)
conv_result = conv_demo(x_demo)
relu_result = torch.relu(conv_result)
flat_result = relu_result.view(1, -1)
logits_demo = fc_demo(flat_result)

# Loss
target_demo = torch.tensor([0])
loss_demo = nn.CrossEntropyLoss()(logits_demo, target_demo)

# Backward
loss_demo.backward()

print(f"Loss: {loss_demo.item():.6f}")
print()

print("Conv layer gradients (∂Loss/∂kernel):")
print(f"  Shape: {conv_demo.weight.grad.shape}")
print(f"  Values:\n{conv_demo.weight.grad.squeeze()}")
print(f"  → ဒီ gradient values က kernel weights ကို update ဖို့ direction ပြောပေးတယ်")
print()

print("FC layer gradients (∂Loss/∂weights):")
print(f"  Shape: {fc_demo.weight.grad.shape}")
print(f"  Values:\n{fc_demo.weight.grad}")
print()

print("⚠️ Gradient = 0 ဆိုရင် → weight ပြောင်းလည်း loss မပြောင်း (dead neuron / plateau)")
print("⚠️ Gradient ကြီးမားရင် → update step ကြီးပြီး overshooting ဖြစ်နိုင်")

=== CNN Layer Gradients ===

Loss: 0.474124

Conv layer gradients (∂Loss/∂kernel):
  Shape: torch.Size([1, 1, 3, 3])
  Values:
tensor([[-0.0378, -0.0378, -0.0378],
        [-0.0378, -0.0378, -0.0379],
        [-0.0379, -0.0379, -0.0379]])
  → ဒီ gradient values က kernel weights ကို update ဖို့ direction ပြောပေးတယ်

FC layer gradients (∂Loss/∂weights):
  Shape: torch.Size([3, 4])
  Values:
tensor([[-1.1327e+01, -1.3215e+01, -1.8878e+01, -2.0766e+01],
        [ 1.3977e-03,  1.6307e-03,  2.3295e-03,  2.5625e-03],
        [ 1.1326e+01,  1.3213e+01,  1.8876e+01,  2.0764e+01]])

⚠️ Gradient = 0 ဆိုရင် → weight ပြောင်းလည်း loss မပြောင်း (dead neuron / plateau)
⚠️ Gradient ကြီးမားရင် → update step ကြီးပြီး overshooting ဖြစ်နိုင်


---

## 4. Gradient Descent & Optimizers — Weights Update

Gradient ရပြီဆိုတော့ weights ကို **Loss ကျအောင်** update လုပ်ရမယ်။

### Basic Gradient Descent

$$w_{\text{new}} = w_{\text{old}} - \text{lr} \times \frac{\partial \text{Loss}}{\partial w}$$

- **lr** (learning rate) = update step size
- Gradient ရဲ့ **ဆန့်ကျင်ဘက်** ဦးတည်ချက်ကို ယူ (gradient descent = downhill)

### Optimizer Variants

| Optimizer | Key Idea | ကောင်းတဲ့ အချက် |
|-----------|----------|---------------|
| **SGD** | Basic gradient descent | Simple, generalizes well |
| **SGD + Momentum** | ယခင် gradient direction ကို မှတ်ထား | Faster convergence, less oscillation |
| **Adam** | Adaptive learning rate per weight | Fast, less sensitive to lr |
| **AdamW** | Adam + weight decay fix | Adam ထက် regularization ပိုကောင်း |

In [ ]:
# --- Gradient Descent: Step-by-Step Example ---
print("=== Manual Gradient Descent ===")
print("Formula: w_new = w_old - lr × gradient\n")

# y = w*x, target=10, x=3 → optimal w = 10/3 ≈ 3.333
lr = 0.01

w_val = 2.0  # starting weight
x_fixed = 3.0
target_fixed = 10.0

print(f"Goal: find w where w×{x_fixed} = {target_fixed} → optimal w = {target_fixed/x_fixed:.4f}")
print(f"Starting: w = {w_val}, lr = {lr}\n")

history = []
for step in range(6):
    y_val = w_val * x_fixed
    loss_val = (y_val - target_fixed) ** 2
    grad_val = 2 * (y_val - target_fixed) * x_fixed  # ∂Loss/∂w
    w_new = w_val - lr * grad_val
    
    history.append((step, w_val, y_val, loss_val, grad_val))
    
    print(f"  Step {step}: w={w_val:.4f} → y={y_val:.4f} → Loss={loss_val:.4f}"
          f" → grad={grad_val:.4f} → w_new={w_new:.4f}")
    w_val = w_new

print(f"\n  w: 2.0 → {w_val:.4f} (optimal: {target_fixed/x_fixed:.4f})")
print(f"  Loss ကျသွားတာ မြင်ရမယ် ✅")

# Plot gradient descent path
steps, ws, ys, losses, grads = zip(*history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, losses, 'r-o', markersize=6)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Decreasing Over Steps")
ax1.grid(True, alpha=0.3)

ax2.plot(steps, ws, 'b-o', markersize=6)
ax2.axhline(y=target_fixed/x_fixed, color='g', linestyle='--', label=f'Optimal w={target_fixed/x_fixed:.3f}')
ax2.set_xlabel("Step")
ax2.set_ylabel("Weight (w)")
ax2.set_title("Weight → Optimal Value") 
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Optimizer Comparison: SGD vs Momentum vs Adam ---
print("=== Optimizer Comparison ===\n")

def train_with_optimizer(optimizer_class, lr, steps=50, **kwargs):
    """Simple 1-weight optimization to compare optimizers"""
    w = nn.Parameter(torch.tensor(0.5))
    opt = optimizer_class([w], lr=lr, **kwargs)
    losses = []
    for _ in range(steps):
        y = w * 3.0
        loss = (y - 10.0) ** 2
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

# Compare
optimizers = {
    'SGD (lr=0.01)':           (optim.SGD,  0.01, {}),
    'SGD+Momentum (lr=0.01)':  (optim.SGD,  0.01, {'momentum': 0.9}),
    'Adam (lr=0.01)':          (optim.Adam, 0.01, {}),
    'AdamW (lr=0.01)':         (optim.AdamW, 0.01, {}),
}

plt.figure(figsize=(12, 5))
for name, (opt_cls, lr, kw) in optimizers.items():
    losses = train_with_optimizer(opt_cls, lr, steps=50, **kw)
    plt.plot(losses, label=name, linewidth=2)

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Optimizer Comparison — Loss ကျတဲ့ Speed ကွာခြားချက်")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print("📊 Adam/AdamW ↓ faster converge (adaptive lr), SGD+Momentum ↓ smooth path")
print("   Plain SGD ↓ slow but sometimes generalizes better with proper tuning")

---

## 5. Learning Rate & Scheduler

### Learning Rate (lr) ရဲ့ အရေးကြီးမှု

$$w_{\text{new}} = w_{\text{old}} - \underbrace{\text{lr}}_{\text{step size}} \times \text{gradient}$$

| LR | Effect |
|----|--------|
| **Too Large** (0.1+) | Loss oscillate / diverge (ပျံသွား) |
| **Too Small** (1e-6) | Converge ဖြစ်ဖို့ epoch အရမ်းများလိုမယ် |
| **Just Right** (1e-3~1e-4) | Smooth convergence ✅ |

### Learning Rate Scheduler

Training ရဲ့ **ပထမပိုင်း**မှာ lr ကြီးကြီးနဲ့ fast learn, **နောက်ပိုင်း**မှာ lr သေးလေးနဲ့ fine-tune:

```
StepLR:          ████████░░░░░░░░  → epoch 10 မှာ lr/2, epoch 20 မှာ lr/4
CosineAnnealing: ████▓▓▒▒░░░░▒▒▓  → cosine curve နဲ့ smooth ကျ
ReduceLROnPlateau: loss မကျတော့ → lr လျှော့
```

In [ ]:
# --- Learning Rate: Too High vs Too Low vs Just Right ---
print("=== Learning Rate Impact ===\n")

def optimize_lr(lr, steps=30):
    w = nn.Parameter(torch.tensor(0.5))
    opt = optim.SGD([w], lr=lr)
    losses = []
    for _ in range(steps):
        y = w * 3.0
        loss = (y - 10.0) ** 2
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(min(loss.item(), 500))  # cap for display
    return losses

lrs = {'lr=0.001 (Too Small)': 0.001, 'lr=0.01 (Good)': 0.01,
       'lr=0.05 (Fast)': 0.05, 'lr=0.12 (Too Large)': 0.12}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, (name, lr) in enumerate(lrs.items()):
    losses = optimize_lr(lr)
    axes[i].plot(losses, 'r-o', markersize=3)
    axes[i].set_title(name, fontsize=11, fontweight='bold')
    axes[i].set_xlabel("Step")
    axes[i].set_ylabel("Loss")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(0, max(losses[:5]) * 1.2 if max(losses[:5]) < 500 else 520)

plt.suptitle("Learning Rate Impact — ကြီးလွန်ရင် ပျံ, ငယ်လွန်ရင် နှေး", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Learning Rate Schedulers ---
print("=== Learning Rate Schedulers ===\n")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = 50

scheduler_configs = [
    ("StepLR (step=15, γ=0.5)", lambda opt: optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)),
    ("CosineAnnealingLR", lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)),
    ("ReduceLROnPlateau", None),  # special handling
]

for idx, (name, sched_fn) in enumerate(scheduler_configs):
    lrs_history = []
    model_dummy = nn.Linear(1, 1)
    opt = optim.Adam(model_dummy.parameters(), lr=0.01)
    
    if name == "ReduceLROnPlateau":
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
        # Simulate: loss ကျပြီးမှ plateau ဖြစ်
        for e in range(epochs):
            fake_loss = max(1.0 - e * 0.05, 0.3) + (0.0 if e < 20 else 0.0)
            sched.step(fake_loss)
            lrs_history.append(opt.param_groups[0]['lr'])
    else:
        sched = sched_fn(opt)
        for e in range(epochs):
            sched.step()
            lrs_history.append(opt.param_groups[0]['lr'])
    
    axes[idx].plot(lrs_history, 'b-', linewidth=2)
    axes[idx].set_title(name, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel("Epoch")
    axes[idx].set_ylabel("Learning Rate")
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("LR Schedulers — Training ရဲ့ နောက်ပိုင်းမှာ LR ကို လျှော့ချ", fontsize=13)
plt.tight_layout()
plt.show()

print("💡 StepLR: ရိုးရှင်း, epoch X တိုင်း lr ကို γ ဖြင့် multiply")
print("💡 CosineAnnealing: smooth decay + warm restart possible")
print("💡 ReduceLROnPlateau: val_loss improve မဖြစ်ရင် lr ကိုလျှော့ — adaptive!")

---

## 6. Overfitting & Regularization

### Overfitting = Model က Training Data ကို **memorize** လုပ်ပြီး unseen data မှာ perform မကောင်း

```
         Loss
  ↑      
  │  ╲    Train Loss ↓↓ (keeps dropping)
  │   ╲___________
  │         ╱  Val Loss ↓ ပြီး ↑ (ပြန်တက်!)  ← Overfitting ✗
  │    ____╱
  │   ╱
  └──────────────── Epoch →
```

### Overfitting ဖြစ်မှန်း ဘယ်လို သိလဲ?
- **Train accuracy ↑↑** but **Val accuracy ↓** (ကွာကြီး)
- Train loss keeps ↓ but Val loss ↑

### Regularization Techniques

| Technique | ဘာလုပ်လဲ | PyTorch |
|-----------|----------|---------|
| **Dropout** | Random neurons ကို off (training time) | `nn.Dropout(0.5)` |
| **Weight Decay (L2)** | Large weights ကို penalize | `optim.Adam(lr=0.001, weight_decay=1e-4)` |
| **Data Augmentation** | Training data variety တိုး | `transforms.RandomCrop, Flip...` |
| **Early Stopping** | Val loss ↑ ရင် training ရပ် | Manual implementation |
| **Batch Normalization** | Activation distribution normalize | `nn.BatchNorm2d()` |

In [ ]:
# --- Dropout: Visual Demo ---
print("=== Dropout ===\n")

dropout = nn.Dropout(p=0.5)  # 50% neurons off

# Training mode: random neurons → 0
dropout.train()
x_drop = torch.ones(1, 10)
drop_out = dropout(x_drop)
print(f"Input:           {x_drop.squeeze().tolist()}")
print(f"Dropout(p=0.5):  {drop_out.squeeze().tolist()}")
print(f"  → Random neurons 0 ဖြစ်သွား (training)")
print(f"  → Active neurons ကို 1/(1-p) = 2x scale up (expected value ထိန်းဖို့)")
print()

# Eval mode: dropout OFF (all neurons active)
dropout.eval()
drop_out_eval = dropout(x_drop)
print(f"Eval mode:       {drop_out_eval.squeeze().tolist()}")
print(f"  → Dropout OFF, all neurons active (inference time)")
print()

# --- Weight Decay: L2 Regularization ---
print("=== Weight Decay (L2 Regularization) ===\n")
print("Total Loss = CrossEntropyLoss + λ × Σ(w²)")
print()
print("Example:")
weights = torch.tensor([0.5, 1.2, -0.8, 2.0])
l2_penalty = (weights ** 2).sum()
lam = 1e-4
print(f"  Weights: {weights.tolist()}")
print(f"  L2 = Σ(w²) = {' + '.join([f'{w.item():.2f}²' for w in weights])} = {l2_penalty.item():.4f}")
print(f"  λ × L2 = {lam} × {l2_penalty.item():.4f} = {lam * l2_penalty.item():.6f}")
print()
print("  → Large weights ကို penalize → weights ကို small ဖြစ်အောင် ထိန်း")
print("  → PyTorch: optimizer = Adam(params, lr=0.001, weight_decay=1e-4)")

In [ ]:
# --- Overfitting vs Good Fit Simulation ---
print("=== Overfitting vs Good Fit ===\n")

np.random.seed(42)
epochs_sim = 50

# Simulate: Good fit
train_loss_good = 2.0 * np.exp(-0.08 * np.arange(epochs_sim)) + 0.1 + np.random.normal(0, 0.02, epochs_sim)
val_loss_good   = 2.0 * np.exp(-0.06 * np.arange(epochs_sim)) + 0.2 + np.random.normal(0, 0.03, epochs_sim)

# Simulate: Overfitting (val loss turns up)
train_loss_overfit = 2.0 * np.exp(-0.1 * np.arange(epochs_sim)) + 0.05 + np.random.normal(0, 0.02, epochs_sim)
val_loss_overfit = np.where(
    np.arange(epochs_sim) < 20,
    2.0 * np.exp(-0.08 * np.arange(epochs_sim)) + 0.3,
    0.3 + 0.015 * (np.arange(epochs_sim) - 20)
) + np.random.normal(0, 0.03, epochs_sim)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_loss_good, 'b-', label='Train Loss', linewidth=2)
ax1.plot(val_loss_good, 'r-', label='Val Loss', linewidth=2)
ax1.set_title("Good Fit ✅\nTrain & Val Loss both ↓", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_loss_overfit, 'b-', label='Train Loss', linewidth=2)
ax2.plot(val_loss_overfit, 'r-', label='Val Loss', linewidth=2)
ax2.axvline(x=20, color='g', linestyle='--', label='← Best checkpoint (Early Stop)')
ax2.set_title("Overfitting ✗\nTrain ↓ but Val ↑ (diverge)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Training Curves: Good Fit vs Overfitting", fontsize=14)
plt.tight_layout()
plt.show()

print("🔑 Overfitting စစ်နည်း: Train Loss ↓ ပေမယ့် Val Loss ↑ ပြန်တက်ရင် = Overfitting")
print("🔑 ဖြေရှင်းနည်း: Dropout, Weight Decay, Data Augmentation, Early Stopping")

---

## 7. Full Training Loop — အဆင့်အကုန် ပေါင်းပြီး Train

Training loop ရဲ့ **exact code flow** ကို line-by-line ရှင်းပြမယ်:

```python
for epoch in range(NUM_EPOCHS):
    # ── TRAINING PHASE ──
    model.train()                          # ① Dropout ON, BatchNorm=batch stats
    for images, labels in train_loader:    # ② Mini-batch ယူ
        images, labels = images.to(device), labels.to(device)  # ③ GPU ပို့
        
        optimizer.zero_grad()              # ④ Previous gradients ရှင်း (⚠️ မဖြစ်မနေ!)
        outputs = model(images)            # ⑤ Forward Pass
        loss = criterion(outputs, labels)  # ⑥ Loss တွက်
        loss.backward()                    # ⑦ Backpropagation (gradients compute)
        optimizer.step()                   # ⑧ Weights Update (gradient descent)
    
    # ── VALIDATION PHASE ──
    model.eval()                           # ⑨ Dropout OFF, BatchNorm=running stats
    with torch.no_grad():                  # ⑩ Gradient computation ပိတ် (memory save)
        for images, labels in val_loader:
            outputs = model(images)
            val_loss = criterion(outputs, labels)
    
    scheduler.step(val_loss)               # ⑪ LR adjust
```

### Critical Steps:
- **④ `zero_grad()`**: ဒါမပါရင် gradients accumulate ဖြစ်ပြီး wrong update ဖြစ်မယ်
- **⑨ `model.eval()`**: ဒါမပါရင် Dropout/BatchNorm training behavior နဲ့ evaluate ဖြစ်ပြီး val accuracy မှားမယ်
- **⑩ `torch.no_grad()`**: Eval time gradient မလို — memory save ရမယ်

In [ ]:
# --- Full Training Loop Demo (Fake Data) ---
# Concepts verify ဖို့ simple model + synthetic data နဲ့ train loop ပြမယ်

print("=== Full Training Loop Demo ===\n")

# --- Synthetic dataset: 2 classes, random images (3x8x8) ---
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
N_train, N_val = 200, 50

# Class 0: mean=0.3, Class 1: mean=0.7
train_x = torch.cat([torch.randn(N_train//2,3,8,8)*0.2 + 0.3,
                      torch.randn(N_train//2,3,8,8)*0.2 + 0.7])
train_y = torch.cat([torch.zeros(N_train//2), torch.ones(N_train//2)]).long()

val_x = torch.cat([torch.randn(N_val//2,3,8,8)*0.2 + 0.3,
                    torch.randn(N_val//2,3,8,8)*0.2 + 0.7])
val_y = torch.cat([torch.zeros(N_val//2), torch.ones(N_val//2)]).long()

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=32, shuffle=False)

# --- Mini CNN ---
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),     # (16, 8, 8)
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.MaxPool2d(2),                      # (16, 4, 4)
    nn.Conv2d(16, 32, 3, padding=1),     # (32, 4, 4)
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),             # (32, 1, 1)
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(32, 2)
)

# --- Training Setup ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight decay = L2
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# --- Training Loop (line-by-line annotated) ---
NUM_EPOCHS = 30
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # ── ① TRAINING PHASE ──
    model.train()                          # Dropout ON, BatchNorm=batch stats
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        optimizer.zero_grad()              # ④ Clear old gradients (MUST DO!)
        outputs = model(images)            # ⑤ Forward pass
        loss = criterion(outputs, labels)  # ⑥ Calculate loss
        loss.backward()                    # ⑦ Backprop: compute gradients
        optimizer.step()                   # ⑧ Update weights
        
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # ── ⑨ VALIDATION PHASE ──
    model.eval()                           # Dropout OFF, BatchNorm=running stats
    running_vloss, vcorrect, vtotal = 0.0, 0, 0
    
    with torch.no_grad():                  # ⑩ No gradient needed for eval
        for images, labels in val_loader:
            outputs = model(images)
            vloss = criterion(outputs, labels)
            running_vloss += vloss.item()
            _, preds = torch.max(outputs, 1)
            vcorrect += (preds == labels).sum().item()
            vtotal += labels.size(0)
    
    val_loss = running_vloss / len(val_loader)
    val_acc = vcorrect / vtotal
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # ⑪ LR scheduler
    scheduler.step(val_loss)
    
    # Best model checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\n✅ Best Val Accuracy: {best_val_acc:.4f} at Epoch {best_epoch+1}")

In [ ]:
# --- Training History Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', label='Train Loss', markersize=3)
ax1.plot(range(1, NUM_EPOCHS+1), val_losses, 'r-o', label='Val Loss', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss per Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train Acc', markersize=3)
ax2.plot(range(1, NUM_EPOCHS+1), val_accs, 'r-o', label='Val Acc', markersize=3)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy per Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Training History — Loss ↓ Accuracy ↑ ဖြစ်ရမယ်", fontsize=13)
plt.tight_layout()
plt.show()

---

## 8. Common Mistakes & Best Practices

### ❌ Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| `zero_grad()` မေ့ | Gradients accumulate → wrong updates | Loop ထဲမှာ `optimizer.zero_grad()` ထည့် |
| `model.eval()` မေ့ | Dropout/BN training behavior ဖြင့် eval | Eval/test time `model.eval()` မဖြစ်မနေ |
| `torch.no_grad()` မေ့ | Eval time memory waste | `with torch.no_grad():` wrapping |
| LR ကြီးလွန်း | Loss diverge / NaN | lr=1e-3 ကနေ start |
| Augmentation on test | Test results biased | Test set: only Resize+ToTensor+Normalize |
| `.to(device)` မေ့ | CPU/GPU mismatch error | Data + model both `.to(device)` |

### ✅ Best Practices

```python
# 1. Reproducibility — seed set
torch.manual_seed(42)
np.random.seed(42)

# 2. Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 3. Training template
for epoch in range(epochs):
    model.train()               # ← MUST
    for batch in train_loader:
        optimizer.zero_grad()   # ← MUST
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    
    model.eval()                # ← MUST
    with torch.no_grad():      # ← MUST
        # validation...

# 4. Save best model
if val_acc > best_val_acc:
    torch.save(model.state_dict(), "best_model.pth")

# 5. Load for inference
model.load_state_dict(torch.load("best_model.pth", weights_only=True))
model.eval()
```

### Training Checklist

- [ ] Data: Train/Val/Test split (eg. 80/10/10)
- [ ] Data: Augmentation on **Train only**
- [ ] Data: `Normalize(mean, std)` on **Train & Test both**
- [ ] Model: `model.to(device)`, `data.to(device)`
- [ ] Loss: `CrossEntropyLoss` for classification
- [ ] Optimizer: `Adam(lr=1e-3)` or `SGD(lr=0.1, momentum=0.9)`
- [ ] Loop: `zero_grad()` → `forward` → `loss` → `backward` → `step`
- [ ] Eval: `model.eval()` + `torch.no_grad()`
- [ ] Monitor: Train/Val loss & accuracy every epoch
- [ ] Save: Best model checkpoint (by val_acc)
- [ ] LR Scheduler: `ReduceLROnPlateau` or `CosineAnnealing`